# Test Stage 9 — A real measurement, emulated: noise, decoupling, servo, and the latency budget

Stage 8 proved the feedback system against a *clean* resonator. No real measurement is
clean. The coil arrives wrapped in its own thermal (Johnson) noise, shaped by the
resonance into a bump; the amplifier adds a flat floor underneath; and the interesting
signals sit in, on, or under all of it. This stage runs the whole system — sweep,
resonance shifting, active feedback — against that reality, with one decisive advantage
over the real coil: **the signal-to-noise ratio (SNR) is a knob**, because the noise is
generated in logic with a software-set amplitude.

```
                          tank noise (Johnson)      amplifier noise
                                  |                        |
generator ─┐                      v                        v
           ├──▶ DAC value ──▶ [resonator] ──▶ (+) ──┬──▶ acquisition system
feedback ──┘                                        │
    ▲                                               │
    └───────────── phase shifter ◀──────────────────┘
```

**SNR, defined precisely for this notebook:** 20·log10 of (coherent tone amplitude ÷
per-sample noise standard deviation), both measured at the acquisition output at the
same settings. Step 1 calibrates the register settings against this definition, so
"repeat at 6 dB" is a reproducible instruction, not a vibe.

## What you need

| | |
|---|---|
| Stages 1–8 | Passing |
| **A bitstream with the noise sources** | Step 0 checks for them and refuses to continue without them |
| Signal generator / cables | None. Everything is internal |
| Time | About ten minutes end to end; the servo step is the longest |

## What the five steps establish

1. **The noise is what it claims to be** — Lorentzian bump at the known f0 with the
   known Q, flat floor beneath, and a floor that falls as sqrt(R): Stage 4's law, now
   proven through the emulated front end. Calibrates the SNR knob.
2. **The phase/gain sweep survives noise** — the Stage 8 sweep repeated at three SNRs,
   quantifying how the extracted loop gain and narrowing phase degrade.
3. **Decoupling works** — a fixed tone plays the electron; feedback in the shifting
   quadrature moves the resonance away, and the tone's amplification must drop by the
   Lorentzian factor the shift predicts.
4. **Active feedback closes the whole loop** — the emulated coil drifts, as real coils
   do with temperature, and a software servo senses the drift and re-steers the
   feedback to hold the operating point. Sense, decide, act, repeat, in noise.
5. **The latency of every stage is measured** — hardware loop, acquisition chain,
   software sense, software act — and summed into the total control interval, which is
   the number that says what this servo can and cannot ever correct.

---

### Simulation or the real trap?

**This is the most emulation-dependent stage in the project**, because the thing it studies —
noise — is generated in logic here and is a fact of nature on the real coil.

Two noise sources stand in for reality: `noise_tank` is injected *into* the resonator, so it
emerges shaped by the resonance exactly as the coil's thermal noise does; `noise_out` is added
*after* it, a flat floor standing in for the amplifier. On the real coil both are simply
there, at whatever level physics and the preamplifier decide.

| | Emulated coil | Real coil |
|---|---|---|
| Noise level | `TANK_SH` / `OUT_SH` set it, in 6 dB steps | Not adjustable. Whatever the coil and amplifier give you |
| Signal-to-noise ratio | `drive_for_snr()` sets the *tone* against a known noise floor | Still works — it drives the generator. But the floor must be measured first, not assumed |
| Drift (Step 4) | Injected on purpose with `reso.shift_f0()` | Real, continuous, and not under your control. The servo is unchanged; only the drift's source differs |
| Where the resonance is | `reso.f0_q`, known exactly | Must be measured. Every use of `reso.f0_q` needs a measured number instead |

Nothing here needs a rebuild. The resonator is off at power-on, so simply not enabling it
leaves the board listening to the real input.


## Setup

> **Real coil — the cell that changes most.** Step 0 below configures the emulated resonator,
> switches it into the signal path and turns its noise sources on. For real measurements,
> leave the resonator disabled: drop `reso.config(...)`, `reso.enable()` and both
> `reso.noise_*()` calls. The board then listens to the actual coil, whose noise arrives on
> its own.


In [ ]:
import os, mmap, struct, time
import numpy as np
import matplotlib.pyplot as plt

FS_CLK = 125e6
F0     = 40.0e6          # the coil being emulated: ~40 MHz
Q_SET  = 180.0           # ~180 at room temperature
SETTLE = 0.02
N_AVG  = 32

# =============================================================================
#  Drivers - the Stage 8 set, with the resonator gaining its two noise knobs.
#  CRITICAL RULE throughout: one 32-bit register access at a time.
# =============================================================================
class Bus(object):
    def __init__(self, base, span=0x1000):
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m  = mmap.mmap(self.fd, span, offset=base)
    def w(self, off, val):
        self.m[off:off + 4] = struct.pack('<I', val & 0xFFFFFFFF)
    def r(self, off):
        return struct.unpack('<I', self.m[off:off + 4])[0]

def s32(v):
    return v - (1 << 32) if v & 0x80000000 else v


class Acq(Bus):
    """Acquisition system as narrowband voltmeter (PEEK at DC) and as
    recorder (capture to memory - defined in the next cell)."""
    def __init__(self):
        Bus.__init__(self, 0x40600000)
        self.dec = 10
    def start(self, dec_log2=10):
        self.dec = dec_log2
        self.w(0x00, 0)
        self.w(0x08, dec_log2)
        self.w(0x0C, 0)
        self.w(0x00, 1 | 2)
        self.w(0x00, 1)
    def tune(self, f):
        self.w(0x04, int(round(f / FS_CLK * (1 << 32))) & 0xFFFFFFFF)
    def measure(self, navg=N_AVG):
        zi = 0.0; zq = 0.0
        for _ in range(navg):
            zi += s32(self.r(0x18))
            zq += s32(self.r(0x1C))
        return complex(zi / navg, zq / navg)
    def f_out(self):
        return FS_CLK / (1 << self.dec)


class Resonator(Bus):
    def __init__(self):
        Bus.__init__(self, 0x40400000)
        self.f0_q = None; self.q_q = None
    def config(self, f0, Q, delay=15, oshift=7):
        w0 = 2.0 * np.pi * f0 / FS_CLK
        r  = 1.0 - np.pi * (f0 / FS_CLK) / Q
        ci = int(round(r * np.cos(w0) * 131072))
        cs = int(round(r * np.sin(w0) * 131072))
        self.w(0x04, ci & 0x3FFFF)
        self.w(0x08, cs & 0x3FFFF)
        self.w(0x0C, delay)
        self.w(0x10, oshift)
        rq        = np.hypot(ci, cs) / 131072.0
        self.f0_q = np.arctan2(cs, ci) / (2.0 * np.pi) * FS_CLK
        self.q_q  = np.pi * (self.f0_q / FS_CLK) / (1.0 - rq)
    def shift_f0(self, df, Q=None):
        """Retune the emulated coil by df hertz - the drift the servo fights.
        One coefficient rewrite; Q is preserved."""
        self.config(self.f0_q + df, Q if Q is not None else self.q_q,
                    delay=15, oshift=7)
    # --- the two noise knobs (registers added for this stage) -------------
    def noise_tank(self, shift=None):
        """Johnson noise INTO the loop -> Lorentzian bump. None = off.
        Each shift step is 6 dB down from raw (~9460 counts rms)."""
        self.w(0x20, 0 if shift is None else (0x100 | (shift & 0x1F)))
    def noise_out(self, shift=None):
        """Amplifier noise AFTER the loop -> flat floor. None = off."""
        self.w(0x24, 0 if shift is None else (0x100 | (shift & 0x1F)))
    def enable(self):  self.w(0x00, 1)
    def disable(self): self.w(0x00, 0)
    def clear(self):   self.w(0x00, 3)
    def clipped(self): return bool(self.r(0x14) & 1)
    def peek(self):    return s32(self.r(0x18)), s32(self.r(0x1C))


class PhaseShift(Bus):
    LATENCY = 2
    def __init__(self):
        Bus.__init__(self, 0x40300000)
    def config_delay(self, f):
        best = None
        for k in range(0, 9):
            D = FS_CLK / (4.0 * f) + k * FS_CLK / f
            if D < 1.0 or D > 1000:
                continue
            frac  = D - np.floor(D)
            score = min(frac, 1.0 - frac)
            if best is None or score < best[0]:
                best = (score, int(np.floor(D)), frac)
        self.w(0x08, best[1])
        self.w(0x0C, int(round(best[2] * 16384)))
        self.w(0x10, 0)
        return best[1], best[2]
    def set(self, A, phi_deg, f=F0):
        phi = phi_deg - 360.0 * self.LATENCY * f / FS_CLK
        self.w(0x00, int(round(A * np.cos(np.radians(phi)) * 16384)) & 0xFFFF)
        self.w(0x04, int(round(A * np.sin(np.radians(phi)) * 16384)) & 0xFFFF)
    def off(self):
        self.set(0.0, 0.0)


class Generator(Bus):
    def __init__(self):
        Bus.__init__(self, 0x40200000, span=0x20000)
    def load_sine(self):
        for i in range(16384):
            v = int(round(8191.0 * np.sin(2.0 * np.pi * i / 16384.0)))
            self.w(0x10000 + 4 * i, v & 0x3FFF)
        self.w(0x08, (16384 << 16) - 1)
        self.w(0x0C, 0)
    def amp(self, frac):
        self.w(0x04, int(round(frac * 8192)) & 0x3FFF)
    def freq(self, f):
        self.w(0x10, int(round(f / FS_CLK * (1 << 30))))
    def run(self):
        self.w(0x00, 0x40)
        self.w(0x00, 0x11)
    def stop(self):
        self.amp(0.0)
        self.w(0x00, 0xC0)


acq, reso, ps, gen = Acq(), Resonator(), PhaseShift(), Generator()
print('drivers ready')

In [ ]:
# =============================================================================
#  Memory capture - the Stage 6/7 system, condensed. Noise SPECTRA need time
#  records, and time records need memory; the PEEK voltmeter cannot see a
#  spectrum. Same safeguards as Stage 7: region from the device tree, stop
#  address inclusive, poison pre-fill, write-error and poison checks that
#  RAISE rather than return plausible garbage. (Bitstreams from the Stage 8
#  rebuild onward carry the trusted badge, so TrustZone no longer matters -
#  but the checks stay, because they cost nothing and catch everything.)
# =============================================================================
def _be32(b, i=0):
    return struct.unpack('>I', b[i * 4:i * 4 + 4])[0]

def _dt_read(path):
    for root in ('/sys/firmware/devicetree/base', '/proc/device-tree'):
        try:
            with open(os.path.join(root, path), 'rb') as fh:
                return fh.read()
        except (IOError, OSError):
            pass
    return None

blob = _dt_read('memory/reg') or _dt_read('memory@0/reg')
assert blob and len(blob) >= 8, 'cannot read the device tree memory node'
declared_end = _be32(blob, 0) + _be32(blob, 1)
assert declared_end < 512 * 1024**2, 'no memory above the declared end - see Stage 6 Step 1'
MEM_START = (declared_end + 0xFFF) & ~0xFFF
MEM_STOP  = MEM_START + (0x00800000 & ~7) - 8       # LAST address, inclusive
POISON    = 0xDEADBEEF
print('capture region 0x%08X .. 0x%08X inclusive' % (MEM_START, MEM_STOP + 7))


CAPTURE_TRACE = None      # set to a path (e.g. '/tmp/capture.log') to log every
                          # capture to disk before it runs


def _trace(msg):
    """Append to the trace file and force it to disk.

    A dead kernel takes buffered stdout with it, which is why a crash inside
    capture() leaves the notebook showing nothing at all. This survives,
    so the last line in the file names the call that killed it."""
    if not CAPTURE_TRACE:
        return
    with open(CAPTURE_TRACE, 'a') as fh:
        fh.write('%.3f %s\n' % (time.time(), msg))
        fh.flush()
        os.fsync(fh.fileno())


def _poison(nbytes):
    fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
    mm = mmap.mmap(fd, nbytes, offset=MEM_START)
    try:
        mm[0:nbytes] = np.full(nbytes // 4, POISON, dtype='<u4').tobytes()
    finally:
        mm.close(); os.close(fd)


def capture(f_lo, dec_log2, n_samples, settle=SETTLE):
    """Record n_samples of the complex stream; raises on any transport fault."""
    _trace('capture f=%.1f dec=%d n=%d' % (f_lo, dec_log2, n_samples))
    _poison(n_samples * 8)
    acq.w(0x00, 0)
    acq.w(0x08, dec_log2); acq.dec = dec_log2
    acq.w(0x0C, 0)
    acq.tune(f_lo)
    acq.w(0x28, MEM_START)
    acq.w(0x2C, MEM_STOP)
    # Read the bounds back BEFORE arming. The writer is a DMA master: if the
    # stop bound did not take - wrong bitstream, unimplemented register, a
    # write that silently went nowhere - it runs from MEM_START upward through
    # whatever follows, and what follows is the memory Linux is using. That is
    # not an exception with a traceback to read; it is a dead kernel. Two
    # register reads cost microseconds and turn the worst failure mode in this
    # notebook into a sentence.
    lo_rb, hi_rb = acq.r(0x28), acq.r(0x2C)
    if lo_rb != MEM_START or hi_rb != MEM_STOP:
        acq.w(0x00, 0)                              # ensure nothing is armed
        raise RuntimeError(
            'capture bounds did not take: wrote 0x%08X..0x%08X, read back '
            '0x%08X..0x%08X. Refusing to arm an unbounded DMA writer.'
            % (MEM_START, MEM_STOP, lo_rb, hi_rb))
    if n_samples * 8 > (MEM_STOP + 8 - MEM_START):
        raise ValueError('record of %d samples (%d bytes) exceeds the %d-byte '
                         'capture region'
                         % (n_samples, n_samples * 8, MEM_STOP + 8 - MEM_START))
    # SETTLE FIRST, RECORD SECOND. Arming immediately and sleeping afterwards
    # would let the record's opening milliseconds hold the filter pipeline still
    # flushing the PREVIOUS tuning - which in a sweep poisons each point with its
    # predecessor's power, most visibly the first point, and can drag a fit wide
    # by tens of percent from the far end of the grid. Instead the chain
    # free-runs at the new tuning while the pipeline flushes, and only then does
    # the writer start: sample 0 of every record is already settled.
    acq.w(0x00, 1 | 2)                              # run at the new tuning...
    acq.w(0x00, 1)
    time.sleep(settle)                              # ...flush the pipeline...
    acq.w(0x00, 1 | 16 | 128)                       # ...then arm, fresh writer
    acq.w(0x00, 1 | 16)
    # WAIT FOR THE WRITER; DO NOT GUESS AT IT.
    #
    # This used to be one sleep of "as long as the record should take, plus
    # 5 ms", after which the writer was disarmed and memory read. Two things
    # were wrong with that, and both showed up on 2026-08-04 as an intermittent
    # 'unwritten words in the record' on captures that were perfectly healthy:
    #
    #  1. DMA_COUNT (0x34) was not consulted until AFTER the decision to stop
    #     had already been made. The sleep was a guess with no verification.
    #  2. DMA_COUNT reports samples HANDED TO the memory writer, not samples
    #     that have landed in DDR. The writer bursts, so the last burst can
    #     still be in flight when the count already reads complete.
    #
    # A fixed 5 ms of margin is also the wrong shape: it is 30% of a short
    # record and 0.1% of a long one, so the failure is rare, load-dependent and
    # looks like a hardware fault. It is not - the record was read a few
    # hundred microseconds early.
    #
    # Poll until the record is complete, then let the last burst land.
    t_record = n_samples * (1 << dec_log2) / FS_CLK
    deadline = time.time() + t_record + 0.5         # generous, not 5 ms
    while int(acq.r(0x34)) < n_samples and time.time() < deadline:
        time.sleep(0.001)
    acq.w(0x00, 1)                                  # disarm
    time.sleep(0.002)                               # last burst reaches DDR

    st  = acq.r(0x38)
    if st & 8:
        raise RuntimeError('memory refused writes - see Stage 6; run check_system.py')
    got = min(int(acq.r(0x34)), n_samples)
    # try/finally, because everything between the open and the close can raise:
    # mmap itself, the slice (SIGBUS on a bad physical address), an allocation
    # failure on a tight board. Without it each failure leaks a file descriptor
    # AND a mapping of /dev/mem - and Stage 10's scan loop swallows per-point
    # exceptions by design, so the leak would repeat silently across hundreds
    # of grid points until the process runs out of descriptors or address
    # space. That surfaces later as an unrelated-looking out-of-memory error.
    def _read_region(nbytes):
        fd = mm = None
        try:
            fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
            mm = mmap.mmap(fd, ((nbytes + 8191) // 4096) * 4096, offset=MEM_START)
            return mm[0:nbytes]
        finally:
            if mm is not None:
                mm.close()
            if fd is not None:
                os.close(fd)

    buf = _read_region(got * 8)
    u32 = np.frombuffer(buf, dtype='<u4')
    _limit = max(8, 0.001 * u32.size)
    if got and int(np.count_nonzero(u32 == POISON)) > _limit:
        # A word still inside the writer's burst buffer is not a lost word. If
        # it is genuinely lost it will still be poison in 20 ms; if it was
        # merely late it will not. Retrying once separates the two, which the
        # single-shot check could not: it reported a transport fault for both.
        time.sleep(0.02)
        buf = _read_region(got * 8)
        u32 = np.frombuffer(buf, dtype='<u4')
        n_bad = int(np.count_nonzero(u32 == POISON))
        if n_bad > _limit:
            raise RuntimeError(
                'unwritten words in the record - transport fault, see Stage 6 '
                '(%d of %d words still unwritten 20 ms after the writer '
                'reported the record complete)' % (n_bad, u32.size))
    raw = np.frombuffer(buf, dtype='<i4').astype(np.float64)
    acq.w(0x00, 1)                                  # back to free-running voltmeter
    _trace('   ok got=%d' % got)
    return raw[0::2] + 1j * raw[1::2]


def welch_psd(z, seg=2048):
    """Averaged power spectrum of a complex record - the standard way to see
    noise: many short transforms averaged beat one long one, because noise in
    a single transform is 100% uncertain no matter how long it is."""
    k = len(z) // seg
    w = np.hanning(seg)
    p = np.zeros(seg)
    for i in range(k):
        p += np.abs(np.fft.fft(z[i * seg:(i + 1) * seg] * w)) ** 2
    f = np.fft.fftfreq(seg, 1.0 / acq.f_out())
    order = np.argsort(f)
    return f[order], p[order] / max(k, 1)


def _lorentz_core(f, p, guard=0.05, minpts=3):
    """Centre, width, peak and BASELINE of a Lorentzian power curve.

    1/p is a parabola in f only for a pure Lorentzian. Any additive floor -
    detector noise, an amplifier pedestal - is absorbed into that parabola and
    inflates the fitted width. That is negligible where the peak towers over
    the floor and severe where it does not, because peak power falls as the
    square of the Q ratio: a floor worth 2% of the open-loop peak is worth 32%
    of a 4x-broadened one. Uncorrected bias, measured: +18% in linewidth at
    Q_eff/Q0 = 1, +26% at 0.5, +153% at 0.25.

    The floor is found by search, not from the wings. For each trial floor:
    subtract it, fit, rebuild the full Lorentzian-plus-floor, and keep the
    trial that reproduces the measured points best. A wing estimate needs
    points beyond ~1.5 linewidths, which a broad line does not have on this
    grid - exactly the case that needs correcting most. The search uses every
    point, so it works at any width. Two passes: coarse over [0, min(p)], then
    fine around the winner.

    Returns (f0, linewidth, peak_above_baseline, baseline).
    """
    f = np.asarray(f, dtype=float)
    p = np.asarray(p, dtype=float)
    pmax = float(np.max(p))
    u = f - f.mean()

    def _try(b):
        q = p - b
        keep = q > guard * q.max()
        if int(np.count_nonzero(keep)) < minpts:
            return None
        A, B, C = np.polyfit(u[keep], 1.0 / q[keep], 2, w=q[keep])
        if A <= 0.0:
            return None                    # parabola opens the wrong way
        u0 = -B / (2.0 * A)
        lw = 2.0 * np.sqrt(max(C / A - u0 ** 2, 1e-30))
        p0 = 1.0 / max(C - A * u0 ** 2, 1e-300)
        f0 = f.mean() + u0
        mod = p0 / (1.0 + (2.0 * (f - f0) / lw) ** 2) + b
        return float(np.sqrt(np.mean(((p - mod) / pmax) ** 2))), (f0, lw, p0, b)

    hi = 0.999 * float(np.min(p))
    best, best_b = _try(0.0), 0.0           # b = 0 is the uncorrected fit
    if hi > 0.0:
        for b in np.linspace(0.0, hi, 17):
            got = _try(b)
            if got is not None and (best is None or got[0] < best[0]):
                best, best_b = got, b
        w = hi / 16.0
        for b in np.linspace(max(best_b - w, 0.0), min(best_b + w, hi), 17):
            got = _try(b)
            if got is not None and (best is None or got[0] < best[0]):
                best, best_b = got, b
    if best is None:
        raise ValueError('fit window too narrow: %d points'
                         % int(np.count_nonzero(p > guard * p.max())))
    f0, lw, p0, base = best[1]
    return f0, lw, p0, base


def fit_lorentzian_psd(f, p, guard=0.05):
    """Centre, width from a power spectrum: 1/p is a parabola (Stage 8), after
    the noise floor has been taken out of p."""
    f0, lw, _, _ = _lorentz_core(f, p, guard=guard, minpts=3)
    return f0, lw

In [ ]:
# =============================================================================
#  STEP 0 - right bitstream, resonator configured, and silence when all
#  noise is off. The presence probe targets the NOISE register specifically:
#  a Stage 8 bitstream has the resonator but answers 0x20 with zero, and
#  running this stage against it would produce noiseless "noise" results.
# =============================================================================
reso.w(0x20, 0x105)
if reso.r(0x20) & 0x1FF != 0x105:
    print('*** THIS BITSTREAM HAS NO NOISE SOURCES ***')
    print('    The resonator answered, but its noise register did not hold a value,')
    print('    so this is the Stage 8 build or older. Rebuild and reload:')
    print('      PC, Vivado Tcl console:  source C:/rp/prj/v0.94/Test_Project/project.ps1 build')
    print('      then BITSTREAM_WORKFLOW.pdf, then check_system.py on the board')
    raise SystemExit('stopping: noise sources absent')
reso.w(0x20, 0)                          # probe cleaned up: noise off

reso.config(F0, Q_SET)
dly, frac = ps.config_delay(F0)
ps.off()
gen.load_sine(); gen.amp(0.0); gen.run()
reso.noise_tank(None); reso.noise_out(None)
reso.enable(); reso.clear()

acq.start(dec_log2=10)
acq.tune(reso.f0_q)
time.sleep(0.2)
z_bg = acq.measure(64)
LW0  = reso.f0_q / reso.q_q              # the known linewidth, ~222 kHz

print('emulated coil : f0 = %.1f Hz, Q = %.3f, linewidth = %.0f Hz' % (reso.f0_q, reso.q_q, LW0))
print('quiet check   : |background| = %.3e, clip = %s' % (abs(z_bg), reso.clipped()))
print()
print('STEP 0:', 'PASS' if not reso.clipped() else 'FAIL')

---
## Step 0b — Localise the crash (run only if the kernel dies in Step 1)

A dying kernel takes buffered output with it, so Step 1 leaves no evidence behind. This
cell writes a trace to disk, forcing each line out before the call it describes, and walks
the Step 1 captures one at a time from smallest to largest.

Run it. If the kernel dies, restart it, re-run the setup cells, then run the **last** cell
in this notebook (`show_trace()`) — the final line of the trace names the exact call that
did it, and the pattern says which of the two causes it was:

- **It dies at one particular record size, every time** — the record is outstripping the
  capture region or the writer's bound. The bound readback now added to `capture()` should
  turn that into an exception first.
- **It dies at the shallowest decimation (R = 256)** — the writer is running faster than
  the region can absorb during the arm window.
- **It dies with `noise_out` on but not off** — the amplifier-floor source, not the
  transport.

If nothing dies here, Step 1 is fine and the fault is elsewhere in the cell.


In [ ]:
CAPTURE_TRACE = '/tmp/stage9_capture.log'
open(CAPTURE_TRACE, 'w').close()

# This cell can be run BEFORE Step 1, which is where the noise levels are
# normally set - so fall back to the same values Step 1 uses.
try:
    TANK_SH, OUT_SH
except NameError:
    TANK_SH, OUT_SH = 3, 4


def show_trace(n=25):
    """Print the tail of the trace. Run this AFTER a kernel restart."""
    try:
        lines = open('/tmp/stage9_capture.log').read().strip().split('\n')
    except IOError:
        print('no trace file - the crash happened before any capture'); return
    print('last %d trace lines (the final one is the call that died):' % min(n, len(lines)))
    for l in lines[-n:]:
        print('   ' + l)


print('capture region : 0x%08X .. 0x%08X  (%.1f MB)'
      % (MEM_START, MEM_STOP + 7, (MEM_STOP + 8 - MEM_START) / 1024.0 ** 2))
print('bound readback : 0x%08X .. 0x%08X' % (acq.r(0x28), acq.r(0x2C)))
print()

# Walk the Step 1 captures smallest-first, so the last survivor brackets the fault.
reso.noise_tank(TANK_SH); reso.noise_out(None)
reso.clear(); time.sleep(0.1)
plan = [(512, 11, 'tank only'), (1024, 11, 'tank only'), (2048, 11, 'tank only'),
        (4096, 11, 'tank only  <- Step 1a uses this'),
        (4096, 10, 'tank only'), (4096, 9, 'tank only'),
        (4096, 8, 'tank only   <- shallowest R in Step 1b')]
for n, dec, note in plan:
    print('  capture n=%-5d R=%-5d %s' % (n, 1 << dec, note), end='')
    import sys; sys.stdout.flush()
    z = capture(reso.f0_q, dec, n)
    print('   -> %d samples, var %.4e' % (len(z), float(np.var(z))))

# Now the same again with the amplifier floor on, as Steps 1b and 1c run it.
print()
reso.noise_tank(None); reso.noise_out(OUT_SH)
for n, dec, note in [(4096, 8, 'floor only'), (4096, 11, 'floor only')]:
    print('  capture n=%-5d R=%-5d %s' % (n, 1 << dec, note), end='')
    import sys; sys.stdout.flush()
    z = capture(reso.f0_q, dec, n)
    print('   -> %d samples, var %.4e' % (len(z), float(np.var(z))))

reso.noise_tank(TANK_SH); reso.noise_out(OUT_SH)
z = capture(reso.f0_q, 10, 4096)
print()
print('  both sources, n=4096 R=1024 -> %d samples, var %.4e'
      % (len(z), float(np.var(z))))
print()
print('STEP 0b: every Step 1 capture survived - the transport is not the fault')
CAPTURE_TRACE = None


---
## Step 1 — Is the noise real noise? Calibrating the SNR knob

Three checks, each with one idea behind it:

**1. Shape — by power sweep.** Point the receiver at a frequency, measure how loud the
static is, move on: loudness against frequency traces the noise bump. Each point is one
number (the average power of a short recording), the receiver's window is narrow and
*identical at every point* — so whatever shape the acquisition chain imposes is the same
constant everywhere and cancels by construction. The sweep spans ±2.5 linewidths, the
whole curve including its tails. The fitted centre and width must match the known f0
and Q.

*Why not just look at one spectrum?* Two earlier attempts did, and both failed
honestly: a single spectrum shows the bump *multiplied by the chain's own filter
response* (its band edges dominate the picture), and dividing two spectra to cancel the
chain drowned the 3 dB of Lorentzian curvature in ~35% statistical noise per point. The
sweep has ~2% noise per point and no chain shape to cancel. Simpler, and two orders of
magnitude more precise — verified by simulation before it was adopted.

**2. Statistics.** The flat amplifier floor must fall as sqrt(R) as the averaging
deepens — Stage 4's law. Fresh noise cannot fail this; periodic or repeating "noise"
cannot pass it.

**3. Amplitude.** Measure the per-sample noise and the tone amplitude per unit drive at
the working point; together they turn "give me x dB SNR" into a computed drive setting.

**The knobs** (top of the next cell): `TANK_SH` / `OUT_SH` set the two noise levels in
6 dB steps; `SWEEP_PTS`, `SWEEP_SPAN`, `SWEEP_DEC`, `SWEEP_N` set how finely, widely,
narrowly and precisely the sweep looks. The defaults are chosen so every number below
has comfortable margin.

> **Real coil.** The *method* transfers completely — measuring the noise bump, checking the
> floor falls as the square root of the averaging, and calibrating a drive setting against the
> measured noise. What does not transfer is 1b's premise that you can switch the tank noise
> off and the amplifier floor on independently. On the real coil, measure the floor with the
> resonance detuned well away instead.


In [ ]:
# ============================ CONFIGURATION ==================================
TANK_SH    = 3       # tank (Johnson) noise: 6 dB quieter per step up
OUT_SH     = 4       # amplifier-floor noise: same scale
SWEEP_PTS  = 21      # how many frequencies the power sweep visits
SWEEP_SPAN = 2.5     # sweep half-width, in linewidths
SWEEP_DEC  = 11      # receiver narrowness: R = 2^11 = 2048 -> ~30 kHz window,
                     #   small next to the 222 kHz linewidth, so the window
                     #   barely smears the curve (<1%)
SWEEP_N    = 4096    # samples per point -> ~2% noise per point
# =============================================================================


def noise_sweep(centre):
    """Trace the noise bump: at each frequency, record briefly and take the
    average power. Returns (fitted centre, fitted width, freqs, powers).
    The receiver window is identical at every point, so the chain's own
    response is a constant scale factor and cannot distort the shape."""
    fs = centre + np.linspace(-SWEEP_SPAN * LW0, SWEEP_SPAN * LW0, SWEEP_PTS)
    p  = np.array([float(np.var(capture(fq, SWEEP_DEC, SWEEP_N))) for fq in fs])
    f0, lw = fit_lorentzian_psd(fs, p, guard=0.10)
    return f0, lw, fs, p


# --- 1a. shape ---------------------------------------------------------------
gen.amp(0.0)
reso.noise_tank(TANK_SH)
reso.noise_out(None)          # floor OFF while measuring the bump: a constant
                              # power offset under every point would bend the
                              # reciprocal-parabola fit
reso.clear(); time.sleep(0.1)
f0_n, lw_n, fsw, psw = noise_sweep(reso.f0_q)

# Free the previous figures before drawing. Each live figure holds its canvas
# and this board has little memory to spare; the first figure of a session is
# also the largest single allocation in the notebook (font cache plus canvas),
# which is why an already-full board tends to die exactly here.
plt.close('all')
plt.figure(figsize=(9, 4.5))
plt.plot((fsw - reso.f0_q) / 1e3, psw / psw.max(), 'o', ms=5, label='measured power')
ffx = np.linspace(fsw[0], fsw[-1], 400)
plt.plot((ffx - reso.f0_q) / 1e3,
         1.0 / (1.0 + (2.0 * (ffx - f0_n) / lw_n) ** 2), '-', lw=1,
         label='Lorentzian fit')
plt.axvline(0, color='r', ls=':', lw=1, label='known f0')
plt.xlabel('frequency - f0 [kHz]'); plt.ylabel('normalised noise power')
plt.title('The noise bump by power sweep: width %.0f Hz vs known %.0f Hz' % (lw_n, LW0))
plt.grid(alpha=0.4); plt.legend(); plt.tight_layout(); plt.show()

print('bump centre : %+.0f Hz from known f0' % (f0_n - reso.f0_q))
print('bump width  : %.0f Hz  vs known %.0f Hz  (%+.1f%%)'
      % (lw_n, LW0, 100.0 * (lw_n / LW0 - 1.0)))
ok_shape = abs(f0_n - reso.f0_q) < 0.1 * LW0 and abs(lw_n / LW0 - 1.0) < 0.10

# --- 1b. the sqrt(R) law on the flat floor -----------------------------------
#  Tank off, floor on: white noise into the decimator, whose whole job is to
#  average it away. Per-sample variance must fall as 1/R (amplitude sqrt(R)).
reso.noise_tank(None); reso.noise_out(OUT_SH)
rs, sig = [], []
for L in (8, 9, 10, 11):
    zz = capture(reso.f0_q, L, 4096)
    rs.append(1 << L); sig.append(np.std(zz))
slope = np.polyfit(np.log(rs), np.log(sig), 1)[0]
print()
print('floor vs R  : slope %.3f on a log-log fit  (ideal -0.5)' % slope)
ok_law = -0.65 < slope < -0.35

# --- 1c. calibrating the SNR knob --------------------------------------------
#  Per-sample noise at the working point (R = 1024, both noises on, as later
#  steps run), and tone amplitude per unit drive: any requested SNR becomes a
#  computed drive setting.
reso.noise_tank(TANK_SH); reso.noise_out(OUT_SH)
zz      = capture(reso.f0_q, 10, 4096)
SIGMA_N = float(np.std(zz))
gen.freq(reso.f0_q); gen.amp(0.10)
time.sleep(SETTLE)
A_TONE_01 = abs(acq.measure(256) - z_bg)
gen.amp(0.0)

def drive_for_snr(snr_db):
    """The generator setting that yields a requested per-sample SNR."""
    want = SIGMA_N * (10.0 ** (snr_db / 20.0))
    return min(0.10 * want / A_TONE_01, 0.95)

print('noise sigma : %.3e per sample at R = 1024' % SIGMA_N)
print('tone gain   : %.3e per unit drive' % (A_TONE_01 / 0.10))
print('the knob    : 40 dB -> drive %.4f   20 dB -> %.4f   6 dB -> %.4f'
      % (drive_for_snr(40), drive_for_snr(20), drive_for_snr(6)))
if drive_for_snr(40) >= 0.95:
    print('WARNING: the 40 dB case is clamped at the generator ceiling - raise')
    print('         TANK_SH by one and re-run this cell')
print()
ok1 = ok_shape and ok_law and not reso.clipped()
print('STEP 1:', 'PASS' if ok1 else 'FAIL - do not trust the noise until this passes')

---
## Step 2 — The phase sweep, three times, at three SNRs

Stage 8's phase sweep extracted the loop gain K and the narrowing phase from clean data.
Here it runs at 40 dB (comfortable), 20 dB (realistic), and 6 dB (hostile), with the
averaging per point increased as the noise rises — 4x more averaging buys back 6 dB, so
the low-SNR sweeps are slower, which is itself part of the result: **the cost of a sweep
on a real noisy coil is measured here, in seconds, before the real coil is ever
connected.**

Two things must hold for a PASS: the *same* K and narrowing phase recovered at every SNR
(the loop physics does not care about noise; only the measurement error grows), and the
fit uncertainties growing roughly as the SNR falls — if they did not, the error bars
would be lying.

One physical effect to watch rather than fear: with feedback narrowing the line, the
noise bump grows taller as well as thinner — feedback modifies the resonator's
*effective temperature*, which is precisely the effect the Herkenhoff system exploits.
The sweep sees it as amplitude; the fits are on frequencies and widths, so it does not
bias them.

> **Real coil.** Unchanged, and one of the strongest checks in the project: it repeats the
> same phase sweep at three signal-to-noise ratios and asks whether the answer is the same
> each time. That needs no known answer at all — only that the instrument be consistent with
> itself.


In [ ]:
A_FB   = 0.15
PHIS   = np.arange(0.0, 360.0, 30.0)         # 12 phases: enough for a sinusoid
F_Q    = reso.f0_q + np.linspace(-2.5 * LW0, 2.5 * LW0, 21)


def measure_amp(f, navg):
    gen.freq(f); acq.tune(f)
    time.sleep(SETTLE)
    return abs(acq.measure(navg) - z_bg)


def fit_lorentzian_amp(f, a):
    f0, lw, _, _ = _lorentz_core(f, a ** 2, guard=0.05, minpts=3)
    return f0, lw


def fit_sinusoid(phi_deg, y):
    c = np.cos(np.radians(phi_deg)); s = np.sin(np.radians(phi_deg))
    M = np.column_stack([np.ones_like(c), c, s])
    m0, mc, ms = np.linalg.solve(M.T.dot(M), M.T.dot(y))
    amp = np.hypot(mc, ms); phi0 = np.degrees(np.arctan2(ms, mc))
    resid = y - (m0 + mc * c + ms * s)
    return m0, amp, phi0, np.sqrt(np.mean(resid ** 2))


CASES = [('40 dB', 40.0, 32), ('20 dB', 20.0, 128), (' 6 dB', 6.0, 512)]
table = []
for label, snr_db, navg in CASES:
    gen.amp(drive_for_snr(snr_db))
    t0 = time.time()
    lws = []
    for phi in PHIS:
        ps.set(A_FB, phi)
        reso.clear()
        a = np.array([measure_amp(f, navg) for f in F_Q])
        if reso.clipped():
            raise RuntimeError('clipped during the %s sweep - lower the drive' % label)
        _, lw_i = fit_lorentzian_amp(F_Q, a)
        lws.append(lw_i)
    ps.off()
    lw0_f, amp_f, phi0_f, rms_f = fit_sinusoid(PHIS, np.array(lws))
    table.append((label, amp_f / lw0_f, (phi0_f + 180.0) % 360.0,
                  rms_f / amp_f, time.time() - t0))
gen.amp(0.0)

print('  SNR     K       narrowing phase   fit rms/mod   sweep time')
for label, K_i, phin_i, relrms, dt in table:
    print('  %s   %.3f       %6.1f deg        %5.2f        %5.1f s'
          % (label, K_i, phin_i, relrms, dt))

K          = table[0][1]                    # the clean values carry forward
PHI_NARROW = table[0][2]
dK   = max(abs(t[1] - K) for t in table[1:])
dPHI = max(min(abs(t[2] - PHI_NARROW), 360 - abs(t[2] - PHI_NARROW)) for t in table[1:])
print()
print('agreement across SNR: K within %.3f, phase within %.1f deg' % (dK, dPHI))
print()
ok2 = (dK < 0.2 * K) and (dPHI < 15.0) and (table[0][3] < 0.15)
print('STEP 2:', 'PASS' if ok2 else 'FAIL - the sweep does not survive noise')

---
## Step 3 — Decoupling: move the resonance away from the electron

In the real trap, the electron and the resonator are coupled when their frequencies
coincide: the coil damps (cools) the electron's motion, and sometimes that is exactly
wrong — a measurement may need the electron left alone. The cure is not to switch
anything off but to **move the resonator's effective centre** with feedback in the
*shifting* quadrature: 90 degrees away from the narrowing phase, feedback changes the
oscillation rate instead of the damping, so the resonance slides sideways with its width
ideally untouched.

Here a fixed weak tone at the resonator's centre plays the electron. Sitting on
resonance, the resonator amplifies it fully. Then the shifting feedback moves the
resonance away, and the tone — whose own frequency has not changed — finds itself off
the resonance's flank, amplified by less. The drop is not a free parameter: it must
equal the Lorentzian rejection at the measured shift,
`1 / sqrt(1 + (2 x shift / linewidth)^2)`.

The gain can be pushed harder here than in narrowing: at the pure-shift phase the
feedback does not touch the damping, so the self-oscillation limit does not bind in the
same way. The linewidth is measured before and after regardless — if it changed by much,
the phase was not the pure-shift quadrature and the numbers say so.

> **Real coil.** Unchanged in method. The shift is produced by feedback, not by the emulation,
> so it happens on a real coil exactly as it does here. Expect a smaller shift in hertz if the
> real coil's linewidth is narrower.


In [ ]:
PHI_SHIFT = (PHI_NARROW + 90.0) % 360.0     # the pure-shift quadrature
A_DEC     = 1.2                              # harder than narrowing could go

SNR_E     = 20.0                             # the "electron" is a modest signal
gen.amp(drive_for_snr(SNR_E)); gen.freq(reso.f0_q)

# --- where is the resonance, and how big is the electron, before ------------
ps.off(); reso.clear()
acq.tune(reso.f0_q)      # POINT THE RECEIVER AT THE TONE.
                         # Step 2's sweeps leave it parked 2.5 linewidths away;
                         # without this line both tone readings were the
                         # averaged-down residue of a rotating phasor - 370x
                         # too small, and pure noise either side of the shift.
time.sleep(SETTLE)
a_on = abs(acq.measure(256) - z_bg)                       # tone ON resonance
gen.amp(0.0)
# Where the resonance IS = where the noise bump is: the power sweep from
# Step 1, which the chain's own shape cannot distort. The flat floor is
# switched off during sweeps for the same reason as in Step 1a.
reso.noise_out(None)
f0_before, lw_before, _, _ = noise_sweep(reso.f0_q)

# --- apply the shift, re-measure both ----------------------------------------
ps.set(A_DEC, PHI_SHIFT); reso.clear(); time.sleep(SETTLE)
f0_after, lw_after, _, _ = noise_sweep(reso.f0_q)         # same grid: the
reso.noise_out(OUT_SH)                                    # shifted bump stays
gen.amp(drive_for_snr(SNR_E))                             # well inside +-2.5 LW
acq.tune(reso.f0_q)                                       # point the receiver
time.sleep(SETTLE)                                        # (same fix as above)
a_off = abs(acq.measure(256) - z_bg)                      # tone, now off-flank
gen.amp(0.0); ps.off()

shift     = f0_after - f0_before
rejection = a_off / a_on
predicted = 1.0 / np.sqrt(1.0 + (2.0 * shift / lw_after) ** 2)

print('resonance centre : %+.0f -> %+.0f Hz   (shifted %.0f Hz = %.2f linewidths)'
      % (f0_before, f0_after, shift, abs(shift) / LW0))
print('linewidth        : %.0f -> %.0f Hz   (%+.0f%% - should be small)'
      % (lw_before, lw_after, 100.0 * (lw_after / lw_before - 1.0)))
print('electron signal  : %.3e -> %.3e   = %.2f of before' % (a_on, a_off, rejection))
print('predicted        : %.2f   (Lorentzian rejection at the measured shift)' % predicted)
print()
ok3 = (abs(shift) > 0.3 * LW0 and
       abs(rejection - predicted) < 0.25 and
       abs(lw_after / lw_before - 1.0) < 0.30 and
       not reso.clipped())
print('STEP 3:', 'PASS' if ok3 else 'FAIL - the resonance did not decouple as predicted')

---
## Step 4 — The servo: a real measurement's shape, end to end

Real coils drift — temperature moves their centre frequency by more than a linewidth
over a day. A real measurement is therefore a loop: *sense* where the resonance is,
*decide* the correction, *act* through the feedback, and repeat forever. This cell runs
that loop against an emulated drift and logs every timestamp, which makes it two things
at once: the demonstration that the system can hold an operating point in noise, and the
measurement floor for Step 5's latency budget.

**Sensing** uses a two-point discriminator: measure the response at f0 +- half a
linewidth. On centre, the two flanks are equal; drift makes one larger. The imbalance is
a signed, nearly linear error signal, and needs only two voltmeter readings per cycle —
the fastest honest way to sense a resonance.

**Acting** uses the shifting quadrature from Step 3: the servo steers the *effective*
centre back onto the target by adjusting the feedback gain, exactly opposing the drift
of the *physical* centre. Sense and act both happen through the same noisy chain the
real experiment would use.

**Passes if** the servo holds the residual error several times smaller than the drift it
is fighting. The drift here is +-25% of a linewidth on a ~30-cycle period — brutal by
real-coil standards, deliberately.

> **Real coil — read this one carefully.** The servo is real and transfers unchanged. The
> *drift it fights* does not: here it is injected deliberately with `reso.shift_f0()` so its
> size and shape are known. On the real coil the drift is thermal and continuous, so delete
> the injection and let the servo work against the genuine article. You lose the known answer
> and keep the servo.


In [ ]:
TARGET   = reso.f0_q                     # hold the resonance HERE
N_CYC    = 60                            # servo cycles
DRIFT_A  = 0.25 * LW0                    # drift amplitude, +-25% of a linewidth
DRIFT_P  = 30.0                          # drift period, in cycles
NAVG_SRV = 64
SNR_SRV  = 20.0

gen.amp(drive_for_snr(SNR_SRV))

# --- calibrate the discriminator: volts of imbalance per hertz of error ------
#  Apply two known feedback shifts, read the imbalance each time: the ratio is
#  the slope. Measured, not derived - the servo then works in real hertz.
def imbalance():
    lo = measure_amp(TARGET - 0.5 * LW0, NAVG_SRV)
    hi = measure_amp(TARGET + 0.5 * LW0, NAVG_SRV)
    return (hi - lo) / (hi + lo)

def act(gain):
    """Steer the effective centre: signed gain in the shifting quadrature."""
    if gain >= 0:
        ps.set(min(gain, 1.8), PHI_SHIFT)
    else:
        ps.set(min(-gain, 1.8), (PHI_SHIFT + 180.0) % 360.0)

ps.off(); reso.clear(); time.sleep(SETTLE)
cal_gain = 0.5
act(+cal_gain); time.sleep(SETTLE); d_plus  = imbalance()
act(-cal_gain); time.sleep(SETTLE); d_minus = imbalance()
act(0.0)
# the same +-gain also moved the centre by a known amount? No - measured:
# Step 3 measured the shift per unit gain, reuse it
HZ_PER_GAIN = abs(shift) / A_DEC
SLOPE = (d_plus - d_minus) / (2.0 * cal_gain * HZ_PER_GAIN)   # imbalance per Hz
print('discriminator: %.3e imbalance per Hz; actuator: %.0f Hz per unit gain'
      % (SLOPE, HZ_PER_GAIN))

# --- the loop, off then on ---------------------------------------------------
def run_servo(closed):
    reso.config(F0, Q_SET)               # start undrifted
    reso.clear(); act(0.0)
    errs, tstamps, sense_t, act_t = [], [], [], []
    gain = 0.0; integ = 0.0
    for n in range(N_CYC):
        drift = DRIFT_A * np.sin(2.0 * np.pi * n / DRIFT_P)
        reso.shift_f0((F0 + drift) - reso.f0_q)      # the coil wanders...
        t0 = time.time()
        err_hz = imbalance() / SLOPE                 # ...the servo senses...
        t1 = time.time()
        if closed:
            # ...decides. PI control: a
            # proportional controller always LAGS a moving target - the first
            # run measured exactly the 2.0x improvement theory predicts for
            # P = 0.4 against this drift. The integral term keeps pushing while
            # any error persists, so slow drift is followed instead of chased.
            # Gains chosen by simulation: 11x nominal improvement, 7.5x even
            # with the slope calibration 30% off. The integrator pauses while
            # the actuator is clamped, so it cannot wind up beyond the rails.
            if abs(gain) < 1.8:
                integ += err_hz
            gain -= (0.5 * err_hz + 0.5 * integ) / HZ_PER_GAIN
            gain  = max(-1.8, min(1.8, gain))
            act(gain)                                # ...and acts
        t2 = time.time()
        errs.append(err_hz); tstamps.append(t0)
        sense_t.append(t1 - t0); act_t.append(t2 - t1)
    act(0.0)
    return np.array(errs), np.array(sense_t), np.array(act_t), np.array(tstamps)

err_open, _,  _,  _   = run_servo(False)
err_clsd, ts, ta, tt  = run_servo(True)
gen.amp(0.0)

skip = int(DRIFT_P / 3)                  # ignore the acquisition transient
rms_open = np.sqrt(np.mean(err_open[skip:] ** 2))
rms_clsd = np.sqrt(np.mean(err_clsd[skip:] ** 2))

# Free the previous figures before drawing. Each live figure holds its canvas
# and this board has little memory to spare; the first figure of a session is
# also the largest single allocation in the notebook (font cache plus canvas),
# which is why an already-full board tends to die exactly here.
plt.close('all')
plt.figure(figsize=(9, 4.5))
plt.plot(err_open / 1e3, 'o-', ms=3, lw=0.7, label='servo OFF: the raw drift')
plt.plot(err_clsd / 1e3, 's-', ms=3, lw=0.7, label='servo ON')
plt.xlabel('servo cycle'); plt.ylabel('centre error [kHz]')
plt.title('Holding the operating point against a drifting coil, in noise')
plt.grid(alpha=0.4); plt.legend(); plt.tight_layout(); plt.show()

CYCLE_T = float(np.median(np.diff(tt)))
print('error rms : servo off %.0f Hz   servo on %.0f Hz   (improvement %.1fx)'
      % (rms_open, rms_clsd, rms_open / max(rms_clsd, 1e-9)))
print('cycle time: median %.0f ms  (sense %.0f ms, act %.1f ms)'
      % (1e3 * CYCLE_T, 1e3 * np.median(ts), 1e3 * np.median(ta)))
print()
ok4 = rms_open / max(rms_clsd, 1e-9) > 3.0
print('STEP 4:', 'PASS' if ok4 else 'FAIL - the servo is not holding the line')

---
## Step 5 — The latency budget: how long between the world changing and the system reacting

Every number here is *measured on this board*, not quoted. The method for the two
hardware latencies uses the memory writer as a stopwatch: the sample counter and the
recorded samples share one index, so "read the counter, poke the stimulus, read the
counter again, then find the edge in the record" brackets the event to a couple of
samples — microsecond-class timing from Python, which by itself can only see
milliseconds.

The stages, from fastest to slowest:

| Stage | What it is |
|---|---|
| Hardware feedback loop | Resonator -> phase shifter -> back. Measured in Stage 1 at 119.87 ns; emulated here by the DELAY register |
| Acquisition chain | Input change -> readable number. Dominated by the correction filter's group delay: 512 taps at the output rate, ~2.1 ms at R = 1024, plus the decimator. The measurement must agree with that arithmetic |
| Software sense | Reading and averaging registers from Python |
| Software decide + act | The servo's arithmetic and the coefficient writes |
| Settle | The resonator rings up in Q/(pi f0) = microseconds; the filter needs its group delay again |

The sum — measured directly as the servo's cycle time in Step 4 — is the **total control
interval**: the shortest time in which this system can sense something and answer it in
software. Everything faster than that must live in the fabric, which is exactly the case
for the still-unbuilt trigger synchronisation, and this table is the quantified argument
for it.

> **Real coil.** Unchanged. This measures how long the *software* takes to see a change and
> respond to it, which is a property of the board and the notebook, not of the plant. The
> numbers here are what make the Stage 10 self-excited oscillator arithmetic work.


In [ ]:
# --- a. how long is one register operation from Python? ----------------------
t0 = time.time()
for _ in range(2000):
    acq.r(0x18)
t_read = (time.time() - t0) / 2000.0
t0 = time.time()
for _ in range(2000):
    acq.w(0x04, acq.r(0x04))
t_write = (time.time() - t0) / 4000.0          # a read and a write per pass
print('register access : read %.1f us, write %.1f us' % (1e6 * t_read, 1e6 * t_write))


# --- b, c. hardware-timestamped step response --------------------------------
def step_latency(prep_fn, step_fn, n=4096):
    """Arm a capture, let it record quietly, apply step_fn mid-record while
    bracketing it with sample-counter reads, and locate the edge. Counter and
    record share one index, so the difference is the chain latency in samples.

    prep_fn restores the OFF state first (without it, every
    repeat after the first found the stimulus already on, 'detected' the edge
    at the counter read itself, and returned the negative of half the
    read-to-read gap - the first run's -0.04 ms). The settle happens BEFORE
    arming, same discipline as capture()."""
    prep_fn()                                  # known OFF state
    _poison(n * 8)
    acq.w(0x00, 0); acq.w(0x08, 10); acq.dec = 10; acq.w(0x0C, 0)
    acq.tune(reso.f0_q)
    acq.w(0x28, MEM_START); acq.w(0x2C, MEM_STOP)
    acq.w(0x00, 1 | 2); acq.w(0x00, 1)
    time.sleep(0.05)                           # settle at the OFF state first
    acq.w(0x00, (1 | 16) | 128); acq.w(0x00, 1 | 16)
    time.sleep(0.008)                          # a fifth of the record, quiet
    c0 = acq.r(0x34); step_fn(); c1 = acq.r(0x34)
    time.sleep(0.008 + n * 1024 / FS_CLK)
    acq.w(0x00, 1)                             # disarm, back to voltmeter
    fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
    mm = mmap.mmap(fd, ((n * 8 + 8191) // 4096) * 4096, offset=MEM_START)
    raw = np.frombuffer(mm[0:n * 8], dtype='<i4').astype(np.float64)
    mm.close(); os.close(fd)
    z   = np.abs(raw[0::2] + 1j * raw[1::2])
    got = min(int(acq.r(0x34)), n)
    z   = z[:got]
    pre  = np.median(z[:len(z) // 5])
    post = np.median(z[-len(z) // 5:])
    thr  = 0.5 * (pre + post)
    rising = post > pre
    hits = np.nonzero((z > thr) if rising else (z < thr))[0]
    hits = hits[hits > (c0 % n)]               # only after the poke
    edge = int(hits[0]) if len(hits) else -1
    return (edge - 0.5 * (c0 + c1)) / acq.f_out()

gen.freq(reso.f0_q); ps.off(); reso.clear()
lat_sense = [step_latency(lambda: gen.amp(0.0),
                          lambda: gen.amp(drive_for_snr(30.0))) for _ in range(5)]
gen.amp(drive_for_snr(30.0))
lat_act   = [step_latency(lambda: ps.off(),
                          lambda: ps.set(A_FB, PHI_NARROW))     for _ in range(5)]
ps.off(); gen.amp(0.0)

lat_sense_ms = 1e3 * float(np.median(lat_sense))
lat_act_ms   = 1e3 * float(np.median(lat_act))
fir_group    = 1e3 * (511 / 2.0) / acq.f_out()   # the analytic prediction

print('chain latency   : stimulus %.2f ms, actuation %.2f ms  (median of 5)'
      % (lat_sense_ms, lat_act_ms))
print('analytic        : filter group delay %.2f ms + decimator ~0.01 ms' % fir_group)

# --- d. the budget -----------------------------------------------------------
sense_sw = 1e3 * float(np.median(ts))            # from the servo logs, Step 4
act_sw   = 1e3 * float(np.median(ta))
print()
print('THE CONTROL INTERVAL, stage by stage (all measured on this board):')
print('  hardware loop         : 0.000120 ms   (119.87 ns, Stage 1; emulated by DELAY)')
print('  resonator ring-up     : ~0.002 ms     (Q_eff / pi f0)')
print('  acquisition chain     : %8.2f ms   (group delay, measured above)' % lat_sense_ms)
print('  software sense        : %8.1f ms   (two discriminator points, Step 4)' % sense_sw)
print('  software decide + act : %8.1f ms   (P control + two writes)' % act_sw)
print('  ------------------------------------')
print('  total control interval: %8.1f ms   (measured directly: servo cycle)' % (1e3 * CYCLE_T))
print()
print('  => software control is good to roughly %.1f Hz of disturbance.' % (1.0 / (2 * np.pi * CYCLE_T)))
print('     Anything faster must be done in the fabric - which is the quantified')
print('     case for the trigger-synchronisation block HANDOVER.md lists as unbuilt.')
print()
ok5 = (abs(lat_sense_ms - fir_group) < 0.5 * fir_group + 0.2) and (1e3 * CYCLE_T <
      4.0 * (lat_sense_ms + sense_sw + act_sw) + 100.0)
print('STEP 5:', 'PASS' if ok5 else 'FAIL - the budget does not add up; find the missing time')

---
## Stage 9 summary

| Step | What it proved |
|---|---|
| 1 | The noise is thermal-shaped and statistically honest: Lorentzian bump at the known f0/Q, flat amplifier floor, sqrt(R) averaging law, SNR knob calibrated in dB |
| 2 | The phase/gain sweep returns the same K and narrowing phase at 40, 20 and 6 dB SNR — with the averaging cost of each measured in seconds |
| 3 | Feedback in the shifting quadrature moves the resonance off the "electron" and the electron's amplification drops by exactly the Lorentzian rejection — decoupling, quantified |
| 4 | A software servo senses, decides and acts through the noisy chain and holds the operating point against a drifting coil |
| 5 | Every latency measured; the total control interval says what software can correct (~Hz) and what must move into the fabric |

**What this means:** the system has now done, in emulation, everything the real
measurement will ask of it — sweep, shift, decouple, hold — at SNRs from comfortable to
hostile, with every answer checked against a value that was *chosen* in advance.

**Next, in order:**

1. **The real coil.** Cable it to IN1/OUT1, disable the resonator (one register write),
   and run Stages 8–9 minus the SNR-knob steps. Nothing else changes — that was the
   point of keeping one measurement interface throughout. Expect the sweep to be the
   calibration that finds the real narrowing phase.
2. **A dynamical electron** (candidate Stage 10): a second, far higher-Q oscillator
   coupled to the resonator, producing the classic noise *dip* and back-action. The
   noise machinery built here is the prerequisite it needed.
3. **Trigger synchronisation in fabric**, now with a measured argument: the software
   control interval is milliseconds; physics that needs microsecond timing needs the
   sequencer.

In [ ]:
# Board back to power-on behaviour: noise off, feedback open, generator held
# in reset, resonator disabled (real inputs pass through again), capture off.
reso.noise_tank(None); reso.noise_out(None)
ps.off()
gen.stop()
reso.disable()
acq.w(0x00, 0)
print('all quiet: noise off, feedback open, generator stopped, resonator disabled')